# 축3. 자가소비 적합성 분석
**서울시 공영주차장 태양광 캐노피 우선입지 선정 연구**

---

## 분석 개요

태양광 캐노피에서 생산된 전력은 한전 계통망으로 연결되어 재분배되는 구조를 가진다.
본 분석에서는 태양광 전력의 **직접 소비 가능한 두 가지 경로**로 분석 범위를 정의한다.

- **주차장 자체 소비**: 조명, CCTV, 출입기, 안내판 등 설비 전력
- **EV 충전기 직접 공급**: 태양광 → 인버터 → 배전반 → EV 충전기 연계

> **핵심 전제**: 이미 EV 충전기가 설치된 주차장에 태양광 캐노피를 설치하면,  
> 기존 충전 인프라와 즉시 연계되어 자가소비율을 높일 수 있다.  
> 근거: 서울시 2021년 공영주차장 592기 EV 충전기 설치 계획 (2022년 상반기 완료),  
> 서울에너지공사 솔라스테이션 태양광-EV 충전 직접 연계 운영 사례

### 분석 대상
- 서울시 공영주차장 **742개소** (축2 전처리 데이터 기준)

### 사용 데이터
| 데이터 | 출처 | 활용 변수 |
|--------|------|----------|
| 공영주차장 전처리 데이터 | 축2 전처리 결과물 | 주차장명, 위경도, 노상/노외 |
| EV 충전소 현황 | 한국환경공단 공공데이터 API | 충전소 좌표 (74,024개) |
| 자치구별 전기차 등록현황 | 서울시 OA-15640 (2026.3 기준) | 자치구별 EV 등록대수 |
| 공영주차장 안내정보 원본 | 서울시 OA-13122 | 운영시간 (HHMM 형식) |

---
## 1. 데이터 로드

In [69]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 전처리 완료 데이터
df = pd.read_csv(
    r'C:\Users\seonu\Documents\DR-project\canopy\data\processed\parking_axis3_preprocessed.csv',
    encoding='utf-8-sig'
)

# EV 충전소 데이터
df_ev = pd.read_csv(
    r'C:\Users\seonu\Documents\DR-project\canopy\data\processed\서울_EV충전소.csv',
    encoding='utf-8-sig'
)

# 태양광 설치 주차장
df_solar = pd.read_csv(
    r'C:\Users\seonu\Documents\DR-project\canopy\data\processed\solar_parking_list.csv',
    encoding='utf-8-sig'
)

print(f"주차장 데이터: {df.shape}")
print(f"EV 충전소: {df_ev.shape}")
print(f"태양광 설치 주차장: {df_solar.shape}")
print(f"\n태양광 컬럼: {df_solar.columns.tolist()}")

주차장 데이터: (742, 11)
EV 충전소: (74024, 37)
태양광 설치 주차장: (74, 4)

태양광 컬럼: ['개소명', '주소', '설치년도', '설치용량_KW']


---
## 2. 실증근거 분석

변수 설계에 앞서 세 가지 실증근거를 확보하여 분석의 논리적 타당성을 보강한다.

### 2.1 실증근거 ① — 태양광 + EV 충전 병행 설치 현황

**목적**: "태양광 캐노피 설치 주차장에 EV 충전기가 실제로 병행 설치되고 있다"는 근거 마련

**방법 A — 데이터 분석**  
태양광 설치 주차장 66개(서울 소재 + 좌표 확보)를 대상으로  
반경 500m 내 EV 충전소 존재 비율을 산출한다.

**방법 B — 리서치**
- 서울에너지공사 솔라스테이션: 태양광 전력 → ESS → EV 충전 직접 공급 운영 중
- 서울지방조달청 설계 가이드라인(2025): 태양광 전력 → EV 충전기 직접 연계 배전 설계 권고

In [ ]:
# 태양광 주차장 주소 확인
print(df_solar[['개소명', '주소']].to_string())

### 2.1.1태양광 주차장 좌표 변환 (카카오 지오코딩 API)
- 태양광 설치 주차장 74개 주소 → 위도/경도 변환
- 카카오 로컬 API 사용

In [ ]:
import requests
import time

KAKAO_API_KEY = '87c1467a133a6d3bf66de89c8c06ad38'

def geocode(address):
    url = 'https://dapi.kakao.com/v2/local/search/address.json'
    headers = {'Authorization': f'KakaoAK {KAKAO_API_KEY}'}
    params = {'query': address}
    
    try:
        res = requests.get(url, headers=headers, params=params)
        result = res.json()
        if result['documents']:
            doc = result['documents'][0]
            return float(doc['y']), float(doc['x'])  # lat, lng
        return None, None
    except:
        return None, None

# 주소 앞에 서울특별시 추가 후 재시도
lats, lngs = [], []
for addr in df_solar['주소']:
    # 서울 주소면 앞에 추가, 고양시 등 타 지역은 그대로
    if '구 ' in addr and '서울' not in addr:
        full_addr = '서울특별시 ' + addr
    else:
        full_addr = addr
    
    lat, lng = geocode(full_addr)
    lats.append(lat)
    lngs.append(lng)
    time.sleep(0.1)

df_solar['lat'] = lats
df_solar['lng'] = lngs

print(f"좌표 변환 완료")
print(f"성공: {df_solar['lat'].notna().sum()}개")
print(f"실패: {df_solar['lat'].isna().sum()}개")
print(df_solar[df_solar['lat'].isna()][['개소명', '주소']])

지도 매핑 안되는 주차장 8개 제외

서울시 소재 태양광 설치 주차장 72개 중 좌표 확보 가능한 66개를 분석 대상으로 사용

In [ ]:
# 좌표 변환 성공한 66개만 사용 (수동 좌표 제외)
df_solar_final = df_solar[
    df_solar['lat'].notna() & 
    ~df_solar['주소'].str.contains('덕양구|하남시', na=False)
].copy()

# 수동으로 넣은 8개 제외 (원래 None이었던 것들)
manual_names = [
    '난지물재생센터주차장(고양시)',
    '도봉햇빛나눔발전소 1호기(초안산 근린공원주차장)',
    '강일공영주차장(견인차보관소)',
    '목사랑시장 고객주차장&공유센터',
    '광암아리수정수센터(주차장)',
    '와룡공영주차장',
    '중구교육지원센터(동화동 공영주차장)',
    '서남물재생센터 물재생체험관 주차장'
]

df_solar_final = df_solar[
    ~df_solar['개소명'].isin(manual_names)
].dropna(subset=['lat', 'lng']).copy()

print(f"최종 분석 대상: {len(df_solar_final)}개")
print(df_solar_final[['개소명', '주소', 'lat', 'lng']].head(5).to_string())

### 태양광 주차장 좌표 변환 결과
- 전체 74개 중 서울 소재 + 좌표 확보: 66개
- 제외: 고양시 1개, 하남시 1개, 주소 불명확 6개
- 카카오 로컬 API 활용

### 2.1.2태양광 설치 주차장 반경 500m 내 EV 충전소 존재 비율
- 태양광 설치 주차장 66개 기준
- Haversine 공식으로 반경 500m 내 EV 충전소 수 집계
- "태양광 설치 주차장의 X%에 EV 충전소가 병행 설치되어 있다" 근거 마련

In [ ]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lng1, lat2, lng2):
    R = 6371000
    lat1, lng1, lat2, lng2 = map(radians, [lat1, lng1, lat2, lng2])
    dlat = lat2 - lat1
    dlng = lng2 - lng1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlng/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

# EV 충전소 좌표
df_ev['lat'] = pd.to_numeric(df_ev['lat'], errors='coerce')
df_ev['lng'] = pd.to_numeric(df_ev['lng'], errors='coerce')
df_ev_clean = df_ev.dropna(subset=['lat', 'lng'])

ev_lat = df_ev_clean['lat'].values
ev_lng = df_ev_clean['lng'].values

# 태양광 주차장별 반경 500m 내 EV 충전소 수 집계
counts = []
for _, row in df_solar_final.iterrows():
    count = sum(
        haversine(row['lat'], row['lng'], lat, lng) <= 500
        for lat, lng in zip(ev_lat, ev_lng)
    )
    counts.append(count)

df_solar_final['ev_충전소_500m'] = counts

# EV 충전소 존재 여부
df_solar_final['ev_병행여부'] = df_solar_final['ev_충전소_500m'] > 0

print(f"태양광 설치 주차장 중 EV 충전소 병행: {df_solar_final['ev_병행여부'].sum()}개")
print(f"비율: {df_solar_final['ev_병행여부'].mean()*100:.1f}%")
print(f"\nEV 충전소 수 분포:")
print(df_solar_final['ev_충전소_500m'].describe())

In [ ]:
fig = go.Figure()

fig.add_trace(go.Bar(
    x=['EV 충전소 있음', 'EV 충전소 없음'],
    y=[df_solar_final['ev_병행여부'].sum(), 
       (~df_solar_final['ev_병행여부']).sum()],
    marker_color=['#4C72B0', '#DD8452'],
    text=[f"{df_solar_final['ev_병행여부'].sum()}개 ({df_solar_final['ev_병행여부'].mean()*100:.1f}%)",
          f"{(~df_solar_final['ev_병행여부']).sum()}개 ({(~df_solar_final['ev_병행여부']).mean()*100:.1f}%)"],
    textposition='outside'
))

fig.update_layout(
    title='태양광 설치 주차장 반경 500m 내 EV 충전소 병행 현황 (n=66)',
    yaxis_title='주차장 수',
    height=450
)

fig.show()

#### 결과 요약
- 태양광 설치 주차장 66개 중 **65개(98.5%)** 의 반경 500m 내 EV 충전소 존재 확인
- 정량 + 정성 근거를 통해 태양광-EV 충전 연계의 현실적 타당성 확보

### 2.1.3 리서치 결과: 태양광 + EV 충전 연계 실제 사례

**1. 서울에너지공사 솔라스테이션**
서울에너지공사는 태양광으로 생산된 전력을 ESS에 저장하여 전기차 충전용 전력으로 공급하는
"솔라스테이션(복합충전소)"을 구축·운영 중임.
출처: 서울에너지공사 공식 홈페이지 (i-se.co.kr)

**2. 서울지방조달청 상계거래 설계 가이드라인 (2025)**
공공 주차장 태양광 의무화(2025.11)에 맞춰 태양광 전력을 EV 충전기와 직접 연계하는
배전 설계 가이드라인 배포. 태양광 전력 → 자가소비 → 잉여분 상계거래 구조를 실제 공공주차장에 적용 중.
출처: 전기신문 (2025.07)

#### 결과 요약
- 태양광 캐노피 전력의 EV 충전 연계는 이미 정책·실무 차원에서 현실화되고 있음


### 2.2 실증근거 ② — EV 충전소 반경 500m 기준 근거

**목적**: 분석 반경 선택의 정당성 확보

**근거**: 전기차 급속충전소 최적 입지 선정 연구(Journal of Climate Change Research, 2022)에서  
국내 다수 선행연구가 사용한 **500m 셀**을 표준 분석 단위로 채택함을 확인

### 2.3 실증근거 ③ — 노외 vs 노상 자체 전력소비 차이

**목적**: 노외 주차장 가중치 부여의 법적 근거 마련

**근거**: 주차장법에 따라 지하식·건축물식 노외주차장은 **상시 조명 유지 의무**가 있음  
(주차구획 최소 10럭스, 출입구 최소 300럭스 이상)

→ 노외 주차장은 조명·환기·정산기 등 설비 전력을 상시 소비하므로  
운영시간이 동일하더라도 노상보다 자체 전력소비가 유의미하게 높다.

---
## 3. 변수 설계

### 3.1 EV 충전소 반경별 분포 분석

**내부 vs 근접 기준 선정 배경**

- 0m를 쓰지 않는 이유:
  - EV 충전기 좌표 vs 공영주차장 좌표는 서로 다른 기준으로 등록 → 10~40m 오차 흔함
  - 공영주차장 면적 자체가 작아도 30~50m, 크면 100m 이상

- 논문/실무 기준: 동일 시설 판단 30~100m / 보행 접근성 300~500m

In [77]:
import pandas as pd
import numpy as np

# EV 충전소 재로드
df_ev = pd.read_csv(
    r'C:\Users\seonu\Documents\DR-project\canopy\data\processed\서울_EV충전소.csv',
    encoding='utf-8-sig'
)
df_ev['lat'] = pd.to_numeric(df_ev['lat'], errors='coerce')
df_ev['lng'] = pd.to_numeric(df_ev['lng'], errors='coerce')
df_ev = df_ev.dropna(subset=['lat', 'lng'])

ev_lat = df_ev['lat'].values
ev_lng = df_ev['lng'].values

print(f"EV 충전소: {len(df_ev)}개")
print(f"lat 범위: {ev_lat.min():.4f} ~ {ev_lat.max():.4f}")
print(f"lng 범위: {ev_lng.min():.4f} ~ {ev_lng.max():.4f}")

# 수서역북 테스트
test_row = df[df['pklt_nm'] == '수서역북 공영주차장'].iloc[0]
count = sum(
    haversine(test_row['lat'], test_row['lot'], lat, lng) <= 30
    for lat, lng in zip(ev_lat, ev_lng)
)
print(f"\n수서역북 30m 이내: {count}개")

EV 충전소: 74024개
lat 범위: 33.0000 ~ 37.9165
lng 범위: 125.0000 ~ 129.3607

수서역북 30m 이내: 4개


**민감도분석으로 거리정하기**

In [87]:
radii = [30, 50, 70, 100, 300, 400, 500]

for r in radii:
    results = []
    for _, row in df.iterrows():
        count = sum(
            haversine(row['lat'], row['lot'], lat, lng) <= r
            for lat, lng in zip(ev_lat, ev_lng)
        )
        results.append(count)
    df[f'ev_{r}m'] = results
    has = (pd.Series(results) > 0).sum()
    mean = pd.Series(results).mean()
    print(f"{r}m — 충전소 있는 주차장: {has}개 ({has/742*100:.1f}%), 평균: {mean:.2f}개")

30m — 충전소 있는 주차장: 58개 (7.8%), 평균: 0.31개
50m — 충전소 있는 주차장: 116개 (15.6%), 평균: 0.87개
70m — 충전소 있는 주차장: 164개 (22.1%), 평균: 1.49개
100m — 충전소 있는 주차장: 318개 (42.9%), 평균: 4.63개
300m — 충전소 있는 주차장: 715개 (96.4%), 평균: 48.16개
400m — 충전소 있는 주차장: 735개 (99.1%), 평균: 88.52개
500m — 충전소 있는 주차장: 735개 (99.1%), 평균: 135.78개


In [89]:
radii = [200]

for r in radii:
    results = []
    for _, row in df.iterrows():
        count = sum(
            haversine(row['lat'], row['lot'], lat, lng) <= r
            for lat, lng in zip(ev_lat, ev_lng)
        )
        results.append(count)
    df[f'ev_{r}m'] = results
    has = (pd.Series(results) > 0).sum()
    mean = pd.Series(results).mean()
    print(f"{r}m — 충전소 있는 주차장: {has}개 ({has/742*100:.1f}%), 평균: {mean:.2f}개")

200m — 충전소 있는 주차장: 631개 (85.0%), 평균: 21.17개


#### 반경별 집계 결과

| 반경 | 충전소 있는 주차장 | 비율 | 평균 충전소 수 | 선정 |
|------|-----------------|------|--------------|------|
| 30m | 58개 | 7.8% | 0.31개 | |
| **50m** | **116개** | **15.6%** | **0.87개** | **✓ 내부 기준** |
| 70m | 164개 | 22.1% | 1.49개 | |
| 100m | 318개 | 42.9% | 4.6개 | |
| **200m** | **631개** | **85.0%** | **21.2개** | **✓ 근접 기준** |
| 300m | 715개 | 96.4% | 48.2개 | |
| 400m | 735개 | 99.1% | 88.5개 | |

**인사이트:**

- 30m는 너무 엄격 (7.8%)
- 50m, 70m가 실제 내부로 볼 수 있는 수준
- 100m를 주차장 바로 인접 및 인근 도로·건물 인근 포함/도보 1분이내인근
- 300m부터 96%로 급증 → 도보 3-5분 이내 생활권/일반적인 근접 반경
- 400m와 500m 차이 없음 

**후보:**

- 내부: 50m
- 근접: 100m

#### 분포확인

In [92]:
# 분포확인
print("=== 50m 분포 ===")
print(df['ev_50m'].describe())
print(f"\n0개: {(df['ev_50m']==0).sum()}개")
print(f"10개 미만: {(df['ev_50m']<10).sum()}개")
print(f"50개 이상: {(df['ev_50m']>=50).sum()}개")
print(f"100개 이상: {(df['ev_50m']>=100).sum()}개")


print("\n=== 100m 분포 ===")
print(df['ev_100m'].describe())
print(f"\n0개: {(df['ev_100m']==0).sum()}개")
print(f"10개 미만: {(df['ev_100m']<10).sum()}개")
print(f"50개 이상: {(df['ev_100m']>=50).sum()}개")
print(f"100개 이상: {(df['ev_100m']>=100).sum()}개")

print("\n=== 200m 분포 ===")
print(df['ev_200m'].describe())
print(f"\n0개: {(df['ev_200m']==0).sum()}개")
print(f"10개 미만: {(df['ev_200m']<10).sum()}개")
print(f"50개 이상: {(df['ev_200m']>=50).sum()}개")
print(f"100개 이상: {(df['ev_200m']>=100).sum()}개")

print("\n=== 300m 분포 ===")
print(df['ev_300m'].describe())
print(f"\n0개: {(df['ev_300m']==0).sum()}개")
print(f"10개 미만: {(df['ev_300m']<10).sum()}개")
print(f"50개 이상: {(df['ev_300m']>=50).sum()}개")
print(f"100개 이상: {(df['ev_300m']>=100).sum()}개")

=== 50m 분포 ===
count    742.000000
mean       0.867925
std        3.540980
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max       62.000000
Name: ev_50m, dtype: float64

0개: 626개
10개 미만: 721개
50개 이상: 1개
100개 이상: 0개

=== 100m 분포 ===
count    742.000000
mean       4.626685
std       11.203768
min        0.000000
25%        0.000000
50%        0.000000
75%        4.000000
max      150.000000
Name: ev_100m, dtype: float64

0개: 424개
10개 미만: 634개
50개 이상: 4개
100개 이상: 2개

=== 200m 분포 ===
count    742.000000
mean      21.172507
std       24.920330
min        0.000000
25%        3.000000
50%       12.000000
75%       34.000000
max      182.000000
Name: ev_200m, dtype: float64

0개: 111개
10개 미만: 331개
50개 이상: 80개
100개 이상: 9개

=== 300m 분포 ===
count    742.000000
mean      48.156334
std       46.555988
min        0.000000
25%       15.000000
50%       35.000000
75%       74.750000
max      293.000000
Name: ev_300m, dtype: float64

0개: 27개
10개 미만: 148개
50개 이상: 270개
1

100m 분포 문제:

- 0개가 424개 (57.1%) — 절반 이상이 0
- 중앙값 0 — 대부분 주차장이 100m 이내 충전소 없음
- 50개 이상 단 4개 — 변별력 거의 없음

=> 즉 100m를 근접으로 쓰면 너무 sparse해서 변수로서 의미가 약함

200m가 더 나음:

- 변별력: 0개가 111개(15%)로 충전 인프라 부족 지역을 더 잘 구분함. 300m는 0개가 27개(3.6%)밖에 안 돼서 거의 다 포함
- 분포 균형: 200m는 10개 미만이 44.6%로 하위권이 넓어 차별화 가능. 300m는 대부분 10개 이상으로 상위 쏠림
- 논리적 거리: 200m는 도보 2~3분 거리로 "인근 충전소"로 더 타이트하게 정의 가능

**요약**

- 100m는 57%가 0이라 대부분 주차장이 같은 점수(0점)를 받음 
- 즉 근접 변수가 실질적으로 작동을 안 함
- 200m는 15%만 0이고 std도 24.9로 주차장 간 차이를 잘 포착함

=> 근접 EV충전소 거리를 200m로 설정

#### 확정
- **내부 기준 50m**: 좌표 오차(10~40m) + 주차장 면적 고려한 현실적 내부 기준
- **근접 기준 200m**: 도보 2~3분 이내, 0개 비율 15%로 변별력 최적 (300m는 0개 3.6%로 변별력 낮음)

### 3.2 최종 변수 구성

| 하위축 | 변수명 | 설명 | 방향 |
|--------|--------|------|------|
| 자체소비 | 운영시간 (가중평균) | (평일×5 + 주말×2) / 7 | ↑ |
| 자체소비 | 노외여부 (0/1) | 노외=1, 노상=0 | ↑ |
| EV충전 | ev_충전기_설치수 (50m) | 주차장 내부 EV 충전기 수 | ↑ |
| EV충전 | ev_200m | 반경 200m 내 EV 충전소 수 | ↑ |
| EV충전 | ev_등록대수 | 자치구별 전기차 등록대수 | ↑ |

---
## 4. 가중치 산정

가중치는 임의로 설정하지 않고 3단계 분석을 통해 도출하였다.

### 4.1 Step 1 — 상관관계 및 다중공선성 분석

변수 간 다중공선성 여부를 피어슨 상관계수로 확인한다.  
상관이 높은 변수를 동시에 사용하면 특정 정보가 중복 반영되어 가중치 설계가 왜곡될 수 있다.

In [93]:
import numpy as np

# 200m 계산 확인
cols = ['ev_충전기_설치수', 'ev_200m', 'ev_등록대수']

# 상관관계
corr = df[cols].corr()
print("=== 상관계수 ===")
print(corr.round(3))

=== 상관계수 ===
            ev_충전기_설치수  ev_200m  ev_등록대수
ev_충전기_설치수       1.000    0.197   -0.005
ev_200m          0.197    1.000    0.039
ev_등록대수         -0.005    0.039    1.000


In [ ]:
# 노상/노외 수치화
df['노외여부'] = df['pklt_knd_nm'].apply(lambda x: 1 if '노외' in str(x) else 0)

# 상관관계 분석
import numpy as np
cols = ['운영시간', '노외여부']
corr = df[cols].corr()
print("=== 상관계수 ===")
print(corr.round(3))

# 기초 통계
print("\n=== 노상/노외별 운영시간 분포 ===")
print(df.groupby('노외여부')['운영시간'].describe())

In [ ]:
# 노상 운영시간 확인
print(df[df['노외여부'] == 0]['운영시간'].value_counts().sort_index())

# 노외 운영시간 확인
print(df[df['노외여부'] == 1]['운영시간'].value_counts().sort_index())

**인사이트:**

운영시간과 주차장 유형(노상/노외)은 상관관계가 0.676으로 상당히 높은 것으로 나타났다. 이는 노외 주차장이 상대적으로 24시간 운영 비중이 높고, 노상 주차장이 시간 제한 운영 비중이 높은 구조가 반영된 결과로 볼 수 있다. 다만 두 변수는 각각 주차장의 시간적 사용 패턴과 공간·구조적 특성을 반영하므로, 우선순위 지수 산정에 모두 포함하여 가중합 형태로 사용하는 것이 타당하다.

#### 결과 해석

| 변수 쌍 | 상관계수 | 해석 |
|---------|---------|------|
| 운영시간 vs 노외여부 | 0.676 | 중간 상관 → 각각 독립적 정보 보유, 둘 다 유지 |
| EV내부 vs EV근접 | 0.197 | 약한 양의 상관 → 독립적 |
| EV내부 vs EV등록 | -0.005 | 거의 독립 |
| EV근접 vs EV등록 | 0.039 | 거의 독립 |

- 자체소비 그룹: r=0.676이지만 노상 24시간 운영 사례(3개) 등 서로를 완전히 대체하지 못함 → 둘 다 유지
- EV충전 그룹: 3개 변수 모두 r<0.2로 다중공선성 없음
- **EV충전 점수가 전반적으로 낮게 분포하는 것은 세 변수 간 독립성이 높기 때문**이다.  
  강남구는 EV 등록 1위지만 내부 충전기 없는 주차장이 많고,  
  영등포동제2공영은 EV 등록이 적은 자치구임에도 내부 충전기 62개로 1위를 달성한다.  
  이는 단일 변수로는 포착하기 어려운 다양한 자가소비 특성을 복합적으로 반영한 결과이다.

### 4.2 Step 2 — 민감도 분석

가중치를 다양하게 바꿔가며 상위 주차장 순위의 안정성을 검증한다.

#### 자체소비

In [ ]:
rom sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df['운영시간_norm'] = scaler.fit_transform(df[['운영시간']])
df['노외여부_norm'] = df['노외여부'].astype(float)  # 이미 0/1

scenarios = {
    '운영시간만 (1.0/0.0)':    [1.0, 0.0],
    '운영시간 우선 (0.7/0.3)': [0.7, 0.3],
    '제안 (0.6/0.4)':          [0.6, 0.4],
    '균등 (0.5/0.5)':          [0.5, 0.5],
    '노외 우선 (0.3/0.7)':     [0.3, 0.7],
}

ranks = {}
for name, weights in scenarios.items():
    score = (df['운영시간_norm'] * weights[0] +
             df['노외여부_norm'] * weights[1])
    ranks[name] = score.rank(ascending=False).astype(int)

rank_df = pd.DataFrame(ranks)
rank_df['pklt_nm'] = df['pklt_nm'].values
rank_df['노외여부'] = df['노외여부'].values
rank_df['운영시간'] = df['운영시간'].values

# 상위 20개 비교
top20 = rank_df.nsmallest(20, '균등 (0.5/0.5)')
print(top20.set_index('pklt_nm').to_string())

24시간 노외 주차장이 너무 많아서 변별력이 없음

In [ ]:
# 24시간 미만 운영 주차장만 필터링해서 순위 변화 확인
non_24h = rank_df[rank_df['운영시간'] < 24].copy()
non_24h_sorted = non_24h.sort_values('균등 (0.5/0.5)')

print(f"24시간 미만 주차장: {len(non_24h)}개")
print("\n=== 시나리오별 순위 변화 (상위 30개) ===")
print(non_24h_sorted.head(30).set_index('pklt_nm')[
    ['운영시간만 (1.0/0.0)', '운영시간 우선 (0.7/0.3)', 
     '제안 (0.6/0.4)', '균등 (0.5/0.5)', '노외 우선 (0.3/0.7)',
     '노외여부', '운영시간']
].to_string())

In [ ]:
# 노상 주차장만 필터링해서 순위 확인
nodong = rank_df[rank_df['노외여부'] == 0].sort_values('균등 (0.5/0.5)')
print(f"노상 주차장: {len(nodong)}개")
print("\n=== 노상 주차장 상위 20개 순위 비교 ===")
print(nodong.head(20).set_index('pklt_nm')[
    ['운영시간만 (1.0/0.0)', '운영시간 우선 (0.7/0.3)',
     '제안 (0.6/0.4)', '균등 (0.5/0.5)', '노외 우선 (0.3/0.7)',
     '노외여부', '운영시간']
].to_string())

#### 핵심 인사이트:
>24시간 노상 주차장(15-가600구간, 원효대교 남단 등)이:
>
>운영시간만 → 193위 (노외 24시간과 동등) 노외 우선 → 534위 (대폭 하락)
>
>즉 노외여부 가중치가 높을수록 24시간 노상 주차장이 불이익을 받음

**결론:**

- 운영시간 0.6 / 노외여부 0.4

**이유:**

- 24시간 노상이 너무 불이익받지 않으면서도 노외 주차장의 설비 우위를 반영함 민감도 분석에서도 순위 변화가 크지 않음

#### 자체소비 하위축 가중치 결정

**변수 구성:**
| 변수 | 가중치 | 근거 |
|------|--------|------|
| 운영시간 (평일×5 + 주말×2) / 7 | 0.6 | 연속 정량 변수, 설비 가동 시간 직접 반영 |
| 노외여부 (노외=1, 노상=0) | 0.4 | 조명·CCTV·정산기 등 설비 전력소비 구조 차이 |

**가중치 산정 근거:**
1. 상관관계 r=0.676 — 두 변수 중복 정보 존재하나 완전하지 않아 둘 다 유지
2. 민감도 분석 — 0.6/0.4에서 24시간 노상 주차장 과도한 불이익 없이 노외 우위 반영
3. 주차장법 — 노외주차장 상시 조명 의무로 노외 자체소비 우위 법적 근거 확보

- 운영시간 0.6 / 노외여부 0.4 채택
- 이유: 24시간 노상 주차장이 과도한 불이익 없이 노외 우위를 합리적으로 반영

#### EV충전

In [95]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df[['ev_내부_norm', 'ev_200m_norm', 'ev_등록_norm']] = scaler.fit_transform(df[cols])

scenarios = {
    '균등 (1/3씩)':          [1/3, 1/3, 1/3],
    '내부중심 (0.5/0.3/0.2)': [0.5, 0.3, 0.2],
    '제안 (0.4/0.3/0.3)':    [0.4, 0.3, 0.3],
    '근접중심 (0.2/0.5/0.3)': [0.2, 0.5, 0.3],
    '수요중심 (0.2/0.3/0.5)': [0.2, 0.3, 0.5],
}

ranks = {}
for name, weights in scenarios.items():
    score = (df['ev_내부_norm'] * weights[0] +
             df['ev_200m_norm'] * weights[1] +
             df['ev_등록_norm'] * weights[2])
    ranks[name] = score.rank(ascending=False).astype(int)

rank_df = pd.DataFrame(ranks)
rank_df['pklt_nm'] = df['pklt_nm'].values

top20 = rank_df.nsmallest(20, '제안 (0.4/0.3/0.3)')
print("=== 상위 20개 시나리오별 순위 ===")
print(top20.set_index('pklt_nm').to_string())

=== 상위 20개 시나리오별 순위 ===
                균등 (1/3씩)  내부중심 (0.5/0.3/0.2)  제안 (0.4/0.3/0.3)  근접중심 (0.2/0.5/0.3)  수요중심 (0.2/0.3/0.5)
pklt_nm                                                                                                
영등포동제2공영                1                   1                 1                   9                  61
잠실역 공영주차장               2                   3                 2                   1                  29
수서역북 공영주차장              7                   5                 3                  17                   5
영희초교 공영주차장              4                  11                 5                   6                   2
일원1동(기계식)               4                  11                 5                   6                   2
일원동맛의거리                 4                  11                 5                   6                   2
탄천제2호 공영주차장             4                  11                 5                   6                   2
영등포동제3공영               31               

**핵심 관찰:**

- 영등포동제2공영 — 내부 충전기 압도적(62개)으로 내부중심/제안에서 1위, 수요중심에서 61위로 폭락
- 잠실역 — 근접중심에서 1위, 내부+근접 균형 잡힌 곳
- 영등포동제3공영 — 내부중심 2위인데 제안에서 8위, 수요중심 97위 → 내부 충전기는 많지만 EV 수요 낮음
- 천호역 — 수요중심에서 91위지만 내부중심 4위 → EV 등록 낮은 자치구

**민감도 분석 결론:**
- 순위가 시나리오마다 꽤 달라지는 편
- 특히 내부 충전기 수가 많은 영등포동제2,3공영이 수요중심에서 크게 떨어짐
- 우리 논리(1안: EV 수요+공급 모두 높은 곳)에 맞는 건 균등~제안 범위

- 균등(1/3씩): 세 변수 동등 반영
- 제안(0.4/0.3/0.3): 내부 충전기 소폭 우위

### EV충전 하위축 가중치 결정

**상관관계 분석 결과:** 3개 변수 모두 r<0.2로 독립적, 다중공선성 없음

**민감도 분석 결과:** 균등~제안 범위에서 순위 안정적

- ev_충전기_설치수 0.4 / ev_200m 0.3 / ev_등록대수 0.3 채택
- 이유: 균등~제안 범위(내부 0.33~0.4)에서 순위 안정적, 내부 충전기가 직접 연계에 가장 확실

**최종 가중치:**
| 변수 | 가중치 | 근거 |
|------|--------|------|
| ev_충전기_설치수 (50m 내부) | 0.4 | 태양광 직접 연계 가장 확실한 소비처 |
| ev_200m (200m 근접) | 0.3 | 도보 2~3분 이내 인근 충전 인프라 |
| ev_등록대수 (자치구별) | 0.3 | 잠재 EV 충전 수요 |

### 4.2-1. EV충전 점수 재정규화

하위축 간 민감도 분석 과정에서 자체소비_점수(0~1)와 EV충전_점수(0~0.56)의
**스케일 불일치** 문제가 확인되었다.

- 자체소비_점수: 운영시간_norm(0~1) + 노외여부(0 or 1) → 자연스럽게 0~1 범위
- EV충전_점수: 세 변수 각각은 0~1이지만, ev_충전기_설치수(50m)의
  84.4%가 0에 몰려있어 가중합 최대값이 0.56에 그침

스케일이 다른 상태에서 0.8/0.2 가중치를 적용하면 의도한 비율대로
작동하지 않으므로, EV충전_점수를 MinMax 재정규화하여 0~1로 통일한다.

| | 재정규화 전 | 재정규화 후 |
|--|-----------|-----------|
| 최대값 | 0.56 | 1.00 |
| 0.1 미만 비율 | 61.2% | 37.6% |
| 0.5 이상 비율 | 0.1% | 10.2% |

In [98]:
print("=== 자체소비 점수 분포 ===")
print(df['자체소비_점수'].describe())
print(f"\n1.0인 주차장: {(df['자체소비_점수'] == 1.0).sum()}개 ({(df['자체소비_점수'] == 1.0).mean()*100:.1f}%)")
print(f"0.9 이상: {(df['자체소비_점수'] >= 0.9).sum()}개 ({(df['자체소비_점수'] >= 0.9).mean()*100:.1f}%)")
print(f"0.5 미만: {(df['자체소비_점수'] < 0.5).sum()}개 ({(df['자체소비_점수'] < 0.5).mean()*100:.1f}%)")

print("\n=== EV충전 점수 분포 ===")
print(df['EV충전_점수'].describe())
print(f"\n0.5 이상: {(df['EV충전_점수'] >= 0.5).sum()}개 ({(df['EV충전_점수'] >= 0.5).mean()*100:.1f}%)")
print(f"0.1 미만: {(df['EV충전_점수'] < 0.1).sum()}개 ({(df['EV충전_점수'] < 0.1).mean()*100:.1f}%)")

=== 자체소비 점수 분포 ===
count    742.000000
mean       0.669563
std        0.382000
min        0.025424
25%        0.244068
50%        1.000000
75%        1.000000
max        1.000000
Name: 자체소비_점수, dtype: float64

1.0인 주차장: 382개 (51.5%)
0.9 이상: 384개 (51.8%)
0.5 미만: 238개 (32.1%)

=== EV충전 점수 분포 ===
count    742.000000
mean       0.111351
std        0.095927
min        0.000000
25%        0.038900
50%        0.082216
75%        0.149631
max        0.559866
Name: EV충전_점수, dtype: float64

0.5 이상: 1개 (0.1%)
0.1 미만: 454개 (61.2%)


In [99]:
# EV충전 점수도 0~1로 재정규화
df['EV충전_점수_norm'] = scaler.fit_transform(df[['EV충전_점수']])

print("=== EV충전 점수 정규화 후 ===")
print(df['EV충전_점수_norm'].describe())
print(f"0.5 이상: {(df['EV충전_점수_norm'] >= 0.5).sum()}개")
print(f"0.1 미만: {(df['EV충전_점수_norm'] < 0.1).sum()}개")

# 축3 재산출
df['축3_자가소비적합성'] = (df['자체소비_점수'] * 0.8 +
                          df['EV충전_점수_norm'] * 0.2)

print(f"\n축3 점수 분포:")
print(df['축3_자가소비적합성'].describe())
print(f"1.0인 주차장: {(df['축3_자가소비적합성'] == 1.0).sum()}개")

=== EV충전 점수 정규화 후 ===
count    742.000000
mean       0.198889
std        0.171340
min        0.000000
25%        0.069480
50%        0.146849
75%        0.267262
max        1.000000
Name: EV충전_점수_norm, dtype: float64
0.5 이상: 76개
0.1 미만: 279개

축3 점수 분포:
count    742.000000
mean       0.575429
std        0.307136
min        0.027763
25%        0.252498
50%        0.801041
75%        0.829370
max        1.000000
Name: 축3_자가소비적합성, dtype: float64
1.0인 주차장: 1개


> **참고**: 재정규화 후에도 0.1 미만 비율이 37.6%로 여전히 높다.  
> 이는 실제 주차장 내부 EV 충전기 설치 비율(15.6%)이 낮은  
> 데이터 현실을 반영한 것으로, 분석의 한계로 명시한다.

### 4.3 Step 3 — 하위축 간 가중치 (논문 근거)

자체소비 점수와 EV충전 점수 간 가중치는 대한전기학회 논문을 근거로 설정하였다.

> 공영주차장 PV 발전량 150.03kWh 중  
> **자체소비 118.23kWh (78.8%)**, EV 충전 31.8kWh (21.2%)  
> — 대한전기학회 논문 (tkiee.org)

→ **자체소비 0.8 / EV충전 0.2** 채택

민감도 분석 결과 0.8/0.2에서 std=0.307로 가장 높은 변별력 확인.

In [100]:
scenarios = {
    '현재 (0.8/0.2)': [0.8, 0.2],
    '절충 (0.7/0.3)': [0.7, 0.3],
    '균등 (0.5/0.5)': [0.5, 0.5],
    'EV중심 (0.4/0.6)': [0.4, 0.6],
}

for name, weights in scenarios.items():
    score = df['자체소비_점수'] * weights[0] + df['EV충전_점수_norm'] * weights[1]
    ones = (score == 1.0).sum()
    median = score.median()
    std = score.std()
    print(f"{name} — 1.0: {ones}개, 중앙값: {median:.3f}, std: {std:.3f}")

현재 (0.8/0.2) — 1.0: 1개, 중앙값: 0.801, std: 0.307
절충 (0.7/0.3) — 1.0: 1개, 중앙값: 0.703, std: 0.272
균등 (0.5/0.5) — 1.0: 1개, 중앙값: 0.516, std: 0.208
EV중심 (0.4/0.6) — 1.0: 1개, 중앙값: 0.425, std: 0.183


**인사이트**
- std가 0.8/0.2에서 0.307로 제일 높음->  변별력이 가장 좋음
- 중앙값이 0.80으로 높은 건 자체소비 점수 자체가 노외 24시간 주차장이 많아서 그런 걸로 보임. 가중치 문제가 아님

**결론:** 논문 기반 0.8/0.2 유지

**이유:**
- std 가장 높음 → 변별력 최대
- 논문 근거(자체소비 79%, EV충전 21%) 유지 가능

### 4.4 최종 가중치 요약

| 단계 | 변수 | 가중치 | 근거 |
|------|------|--------|------|
| 자체소비 점수 | 운영시간 | 0.6 | 연속 정량 변수, 더 많은 정보량 |
| 자체소비 점수 | 노외여부 | 0.4 | 주차장법 상시 조명 의무 |
| EV충전 점수 | ev_충전기_설치수 (50m) | 0.4 | 태양광 직접 연계 가장 확실 |
| EV충전 점수 | ev_200m (근접) | 0.3 | 도보 2~3분 이내 인근 인프라 |
| EV충전 점수 | ev_등록대수 | 0.3 | 잠재 EV 충전 수요 |
| **축3 최종** | 자체소비 점수 | **0.8** | 논문: 태양광 자체소비 78.8% |
| **축3 최종** | EV충전 점수 (정규화) | **0.2** | 논문: EV 충전 21.2% |

---
## 5. 축3 점수 산출

### 5.1 점수화 공식

```
운영시간 = (평일운영시간 × 5 + 주말운영시간 × 2) / 7

자체소비_점수 = MinMax정규화(운영시간) × 0.6 + 노외여부 × 0.4

EV충전_점수 = (MinMax정규화(ev_충전기_설치수) × 0.4
             + MinMax정규화(ev_200m) × 0.3
             + MinMax정규화(ev_등록대수) × 0.3)

EV충전_점수_norm = MinMax정규화(EV충전_점수)  # 스케일 통일

축3_자가소비적합성 = 자체소비_점수 × 0.8 + EV충전_점수_norm × 0.2
```

In [101]:
# 최종 확정
df['축3_자가소비적합성'] = (df['자체소비_점수'] * 0.8 +
                          df['EV충전_점수_norm'] * 0.2)

print(df[['pklt_nm', 'pklt_knd_nm', '자체소비_점수', 'EV충전_점수_norm', '축3_자가소비적합성']]
      .sort_values('축3_자가소비적합성', ascending=False)
      .head(15).to_string())

            pklt_nm pklt_knd_nm  자체소비_점수  EV충전_점수_norm  축3_자가소비적합성
522        영등포동제2공영      노외 주차장      1.0      1.000000    1.000000
588       잠실역 공영주차장      노외 주차장      1.0      0.771839    0.954368
415      수서역북 공영주차장      노외 주차장      1.0      0.725189    0.945038
665     탄천제2호 공영주차장      노외 주차장      1.0      0.703662    0.940732
529      영희초교 공영주차장      노외 주차장      1.0      0.703662    0.940732
523        영등포동제3공영      노외 주차장      1.0      0.685415    0.937083
188  도곡로21길 7 공영주차장      노외 주차장      1.0      0.665387    0.933077
500   역삼1문화센터 공영주차장      노외 주차장      1.0      0.665387    0.933077
186   도곡로 327 공영주차장      노외 주차장      1.0      0.665387    0.933077
502  역삼문화공원제1호공영주차장      노외 주차장      1.0      0.665387    0.933077
501    역삼문화공원 공영주차장      노외 주차장      1.0      0.665387    0.933077
147      논현초교 공영주차장      노외 주차장      1.0      0.621224    0.924245
681       학여울 공영주차장      노외 주차장      1.0      0.611885    0.922377
646       천호역 공영주차장      노외 주차장      1.0      0.603446    0.92

#### 점수 분포 요약

| 통계량 | 자체소비 점수 | EV충전 점수(정규화) | 축3 최종 점수 |
|--------|------------|-----------------|------------|
| 평균 | 0.670 | 0.199 | 0.575 |
| 표준편차 | 0.382 | 0.171 | 0.307 |
| 중앙값 | 1.000 | 0.147 | 0.801 |
| 최댓값 | 1.000 | 1.000 | 1.000 |

---
## 6. 시각화 및 인사이트

### 6.1 축3 점수 지도

In [104]:
import plotly.graph_objects as go

# 저장
output_cols = ['pklt_nm', 'pklt_knd_nm', 'lat', 'lot', '자치구',
               '운영시간', '노외여부', 'ev_충전기_설치수', 'ev_200m', 'ev_등록대수',
               '자체소비_점수', 'EV충전_점수_norm', '축3_자가소비적합성']

df_result = df[output_cols].copy()
df_result.to_csv(
    r'C:\Users\seonu\Documents\DR-project\canopy\data\processed\parking_axis3_scored.csv',
    index=False, encoding='utf-8-sig'
)
print(f"저장 완료: {len(df_result)}개")
print(f"\n축3 점수 분포:")
print(df_result['축3_자가소비적합성'].describe())

# 지도 시각화
fig = go.Figure()
fig.add_trace(go.Scattermap(
    lat=df_result['lat'],
    lon=df_result['lot'],
    mode='markers',
    marker=dict(
        size=8,
        color=df_result['축3_자가소비적합성'],
        colorscale='Blues',
        showscale=True,
        colorbar=dict(title='축3 점수'),
        cmin=0, cmax=1
    ),
    text=df_result['pklt_nm'] + '<br>' +
         '자치구: ' + df_result['자치구'] + '<br>' +
         '축3 점수: ' + df_result['축3_자가소비적합성'].round(3).astype(str) + '<br>' +
         '자체소비_점수: ' + df_result['자체소비_점수'].round(3).astype(str) + '<br>' +
         'EV 내부(50m): ' + df_result['ev_충전기_설치수'].astype(str) + '개<br>' +
         'EV 근접(200m): ' + df_result['ev_200m'].astype(str) + '개',
    hovertemplate='%{text}<extra></extra>'
))

fig.update_layout(
    map=dict(style='carto-positron',
             center=dict(lat=37.5665, lon=126.9780), zoom=10),
    title='축3 자가소비 적합성 점수 지도 (742개) — 최종',
    height=600
)

fig.show()
fig.write_html(
    r'C:\Users\seonu\Documents\DR-project\canopy\output\figures\축3_점수_지도_최종.html'
)

저장 완료: 742개

축3 점수 분포:
count    742.000000
mean       0.575429
std        0.307136
min        0.027763
25%        0.252498
50%        0.801041
75%        0.829370
max        1.000000
Name: 축3_자가소비적합성, dtype: float64


### 6.2 자체소비 vs EV충전 산점도 (4분면 분석)

- **최우선(우상단)**: 자체소비↑ + EV충전↑ → 영등포동제2공영, 잠실역, 수서역북
- **자체소비 강점(우하단)**: 24시간 노외 주차장 다수 → 자체 설비 중심
- **EV충전 강점(좌상단)**: EV 수요 높은 지역 소형 주차장
- **낮은 적합성(좌하단)**: 노상 단기 운영 → 우선순위 후순위

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 자체소비 vs EV충전 산점도 (4분면)
fig1 = go.Figure()

# 4분면 기준선
mid_x = df_result['자체소비_점수'].median()
mid_y = df_result['EV충전_점수_norm'].median()

colors = df_result['축3_자가소비적합성']

fig1.add_trace(go.Scatter(
    x=df_result['자체소비_점수'],
    y=df_result['EV충전_점수_norm'],
    mode='markers',
    marker=dict(
        size=6,
        color=colors,
        colorscale='Blues',
        showscale=True,
        colorbar=dict(title='축3 점수'),
    ),
    text=df_result['pklt_nm'] + '<br>자체소비: ' + df_result['자체소비_점수'].round(3).astype(str) +
         '<br>EV충전: ' + df_result['EV충전_점수_norm'].round(3).astype(str),
    hovertemplate='%{text}<extra></extra>'
))

# 기준선
fig1.add_hline(y=mid_y, line_dash='dash', line_color='gray', opacity=0.5)
fig1.add_vline(x=mid_x, line_dash='dash', line_color='gray', opacity=0.5)

# 4분면 레이블
fig1.add_annotation(x=0.9, y=0.9, text='★ 최우선<br>(자체소비↑ EV↑)', showarrow=False, font=dict(size=10, color='#1a5fa8'))
fig1.add_annotation(x=0.1, y=0.9, text='EV충전 강점<br>(자체소비↓ EV↑)', showarrow=False, font=dict(size=10, color='gray'))
fig1.add_annotation(x=0.9, y=0.05, text='자체소비 강점<br>(자체소비↑ EV↓)', showarrow=False, font=dict(size=10, color='gray'))
fig1.add_annotation(x=0.1, y=0.05, text='낮은 적합성<br>(자체소비↓ EV↓)', showarrow=False, font=dict(size=10, color='#ccc'))

fig1.update_layout(
    title='자체소비 점수 vs EV충전 점수 산점도',
    xaxis_title='자체소비_점수',
    yaxis_title='EV충전_점수_norm',
    height=500
)
fig1.show()
fig1.write_html(
    r'C:\Users\seonu\Documents\DR-project\canopy\output\figures\자체소비 vs EV충전 산점도.html'
)

### 6.3 자치구별 평균 축3 점수

- 상위: 성동구(0.805), 성북구(0.802), 은평구(0.794)
- 하위: 영등포구(0.343), 노원구(0.347), 금천구(0.354)
- 강남구(0.626) 중간 — EV 등록 1위지만 노상 주차장 비율 영향

In [ ]:
# 자치구별 평균 축3 점수
gu_mean = df_result.groupby('자치구')['축3_자가소비적합성'].mean().sort_values(ascending=True)

fig2 = go.Figure()
fig2.add_trace(go.Bar(
    y=gu_mean.index,
    x=gu_mean.values,
    orientation='h',
    marker=dict(
        color=gu_mean.values,
        colorscale='Blues',
        showscale=True,
        colorbar=dict(title='평균 점수')
    ),
    text=gu_mean.round(3).values,
    textposition='outside'
))

fig2.update_layout(
    title='자치구별 평균 축3 자가소비 적합성 점수',
    xaxis_title='평균 점수',
    yaxis_title='자치구',
    height=700,
    xaxis=dict(range=[0, 1])
)
fig2.show()
fig2.write_html(
    r'C:\Users\seonu\Documents\DR-project\canopy\output\figures\자치구별 평균 축3 점수.html'
)

### 6.4 노상/노외별 점수 분포

- 노외 중앙값 0.80 vs 노상 중앙값 0.15 → 명확한 차이
- 노외여부 가중치(0.4) 부여가 실질적으로 유의미한 차이를 만들어냄

In [ ]:
# 노상/노외별 점수 분포 박스플롯
fig3 = go.Figure()

for knd, color in [('노외 주차장', '#1a5fa8'), ('노상 주차장', '#DD8452')]:
    subset = df_result[df_result['pklt_knd_nm'] == knd]
    fig3.add_trace(go.Box(
        y=subset['축3_자가소비적합성'],
        name=knd,
        marker_color=color,
        boxpoints='outliers',
        jitter=0.3,
        text=subset['pklt_nm'],
        hovertemplate='%{text}<br>점수: %{y:.3f}<extra></extra>'
    ))

fig3.update_layout(
    title='노상/노외별 축3 자가소비 적합성 점수 분포',
    yaxis_title='축3 점수',
    height=500,
    showlegend=True
)
fig3.show()
fig3.write_html(
    r'C:\Users\seonu\Documents\DR-project\canopy\output\figures\노상_노외별_축3_점수_분포.html'
)

### 6.5 상위 20개 주차장 상세 비교

- 영등포동제2공영: EV충전 점수 압도적(내부 62개)
- 대부분 자체소비 1.0 + EV충전 차이로 순위 결정
- 신방화역, 용문동: 자체소비 점수로 상위권 진입

In [ ]:
# 상위 20개 주차장 상세 비교
top20 = df_result.sort_values('축3_자가소비적합성', ascending=False).head(20)

fig4 = go.Figure()

fig4.add_trace(go.Bar(
    name='자체소비 점수',
    y=top20['pklt_nm'],
    x=top20['자체소비_점수'],
    orientation='h',
    marker_color='#1a5fa8',
    opacity=0.8
))

fig4.add_trace(go.Bar(
    name='EV충전 점수',
    y=top20['pklt_nm'],
    x=top20['EV충전_점수_norm'],
    orientation='h',
    marker_color='#FF9800',
    opacity=0.8
))

fig4.add_trace(go.Scatter(
    name='축3 최종점수',
    y=top20['pklt_nm'],
    x=top20['축3_자가소비적합성'],
    mode='markers',
    marker=dict(size=10, color='#E74C3C', symbol='diamond'),
))

fig4.update_layout(
    title='상위 20개 주차장 축3 변수별 상세 비교',
    xaxis_title='점수',
    barmode='overlay',
    height=650,
    legend=dict(x=0.7, y=0.1),
    yaxis=dict(autorange='reversed')
)
fig4.show()
fig4.write_html(
    r'C:\Users\seonu\Documents\DR-project\canopy\output\figures\상위 20개 주차장 축3 변수별 상세 비교.html'
)

---
## 7. 한계 및 향후 과제

| 한계 | 내용 | 향후 보완 방법 |
|------|------|--------------|
| 전력사용량 미공개 | 주차장 실제 전력소비량 없어 운영시간을 대리변수 사용 | 서울시설공단 정보공개청구 |
| 이용률 미공개 | 주차장 실제 이용률 없어 EV 수요 직접 추정 불가 | 서울시설공단 정보공개청구 |
| EV 충전기 내부 판별 한계 | 좌표 오차로 50m 기준이 완전한 내부 보장 불가 | 충전기 시설 ID-주차장 ID 매핑 구축 |
| 자치구 단위 EV 등록대수 | 주차장 단위 수요가 아닌 자치구 평균값 사용 | 행정동 단위 데이터 확보 시 정밀화 |
| 하위축 간 가중치 일반화 | 0.8/0.2는 논문 시뮬레이션 기반, 민감도 분석으로 순위 안정성 검증 완료. 서울시 공영주차장 전반 일반화는 실측 데이터 확보 시 추가 검증 필요 | 실측 발전·소비 데이터 확보 시 보완 |

---
*분석 파일: `03_analysis_axis3.ipynb` | 담당: 이선우 | 최종 수정: 2026.04*